<a href="https://colab.research.google.com/github/tougheye/Data_processing/blob/main/UPTE_Payscale_TCS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

This code would take in Job Summary Reports received from UC and convert them into tables for each campus and job titles producing the step tables.

In [23]:
import pandas as pd

represented_job_codes_df = pd.read_excel('/content/UPTE-26-015_Job Summary Report_3.20.26.xlsx',
                                         sheet_name="Represented Job Codes")
represented_job_codes_df.shape

(121114, 30)

In [24]:
represented_job_codes_df.columns

Index(['Salary Plan SETID', 'Status - Job Code', 'Job Code',
       'Job Code Description', 'Eff Date - Salary Grade',
       'Salary Grade Eff Status', 'Union Code', 'FLSA Status',
       'Personnel Program/CLASS_INDC', 'OnCall Eligible', 'Elig Shift Diff',
       'Sal Plan', 'Sal Plan Description', 'Grade', 'Rate Code', 'Comp Freq',
       'UCPATH Step', 'UC  Half Step', 'Hrly Rate', 'Monthly Rt', 'Annual Rt',
       'CTO/OS Subgroup', 'CTO Description', 'SOC Code',
       'SOC Code Description', 'Census/OCC Code',
       'Census/OCC Code Description', 'EEO1 Reporting Code',
       'EEO1 Reporting Cod Description', 'OSHPD Code'],
      dtype='object')

Filter the large dataset for UPTE bargaining unit titles

In [ ]:
upte_represented_job_codes_df = represented_job_codes_df[represented_job_codes_df['Union Code'].isin(['HX', 'RX', 'TX'])]
upte_represented_job_codes_df.shape


(46098, 30)

In [ ]:
# Check number of unique job codes
upte_represented_job_codes_df['Job Code'].nunique()

574

In [ ]:
cols_to_keep = ['Salary Plan SETID', 'Job Code', 'Job Code Description', 'Union Code','FLSA Status', 'OnCall Eligible', 'Elig Shift Diff',
                'Rate Code', 'UCPATH Step', 'UC  Half Step', 'Hrly Rate', 'Monthly Rt', 'Annual Rt', 'Grade', 'Rate Code',
                'Eff Date - Salary Grade', 'Salary Grade Eff Status']
#

Keep only the necessary columns -


1.   'Salary Plan SETID' - Excel file name
2.   'Job Code'
3.   'Job Code Description'    - Second level grouping
1.   'Union Code'.        - Tab name in Excel file
2.   'FLSA Status'        - In Group label for the second level
4.   'OnCall Eligible'.   - In Group label for the second level
5.   'Elig Shift Diff'.   - In Group label for the second level
6.   'Rate Code'.         - In Group label for the second level
7.   'Hrly Rate'          - transpose to column
8.   'Monthly Rt'         - transpose to column
9.   'Annual Rt'          - transpose to column
10.  'Short Descr'.       - First level grouping in tab
11.  'UCPATH Step'        - column label  
12.  'UC  Half Step'      - column label 2  





In [ ]:
upte_represented_job_codes_refined_df = upte_represented_job_codes_df[cols_to_keep]
# upte_represented_job_codes_refined_df.columns

In [ ]:
business_unit_list = upte_represented_job_codes_df['Salary Plan SETID'].unique()
# business_unit_list

In [ ]:
import math
# dictionary to store the job details
job_details_dict = {}

for business_unit in business_unit_list:
  job_details_dict[business_unit] = {}
  #define the current business unit
  current_bus_unit = upte_represented_job_codes_refined_df[
      upte_represented_job_codes_refined_df['Salary Plan SETID'] == business_unit]

  # loop through the bargaining units
  for bargaining_unit in current_bus_unit['Union Code'].unique():
    job_details_dict[business_unit][bargaining_unit] = {}

    # define the current bargaining unit
    current_bus_unit_bu_df = current_bus_unit[current_bus_unit['Union Code'] == bargaining_unit]

    title_list = sorted(current_bus_unit_bu_df['Job Code Description'].unique())

      # loop through the job_code in the current business unit
    for title in title_list:
      step_col = current_bus_unit_bu_df[current_bus_unit_bu_df['Job Code Description'] == title]['UCPATH Step'].unique()
      step_list = sorted([ float(k) if not math.isnan(k) else 0 for k in step_col])
      half_step_col = current_bus_unit_bu_df[current_bus_unit_bu_df['Job Code Description'] == title]['UC  Half Step'].unique()
      half_step_list = sorted([ float(k) if not math.isnan(k) else 0 for k in half_step_col])
      # create a check valve to indicate if the half steps are the same as the steps
      half_step_indicat = 'N' if step_list == half_step_list else 'Y'
      job_details_dict[business_unit][bargaining_unit][title] = {
            'Title' : title,
            'Job Code' : current_bus_unit_bu_df[current_bus_unit_bu_df['Job Code Description'] == title]['Job Code'].unique(),
            'Exempt Status' : current_bus_unit_bu_df[current_bus_unit_bu_df['Job Code Description'] == title]['FLSA Status'].unique(),
            'OnCall Eligibility Status' : current_bus_unit_bu_df[current_bus_unit_bu_df['Job Code Description'] == title]['OnCall Eligible'].unique(),
            'Shift Differential Status' : current_bus_unit_bu_df[current_bus_unit_bu_df['Job Code Description'] == title]['Elig Shift Diff'].unique(),
            'Steps' : step_list,
            'Half Steps' : half_step_list,
            'Half Steps Status' : half_step_indicat,
            'Hourly Rates': sorted(current_bus_unit_bu_df[current_bus_unit_bu_df['Job Code Description'] == title]['Hrly Rate'].unique()),
            'Monthly Rates': sorted(current_bus_unit_bu_df[current_bus_unit_bu_df['Job Code Description'] == title]['Monthly Rt'].unique()),
            'Annual Rates': sorted(current_bus_unit_bu_df[current_bus_unit_bu_df['Job Code Description'] == title]['Annual Rt'].unique())
    }



In [ ]:
job_details_dict

create 18 Dataframe structures and write them to EXCEL with separate tabs for each bargaining unit

In [ ]:
# # take out UCOP and LBNL to prevent the code from crashing due to not having "HX" in those business units
campus_bu_list = [bus_unit for bus_unit in business_unit_list if bus_unit not in ['UCOP1', 'LBNL1']]

# create dictionary to capture the titles that have duplicate steps with different pay rates
manual_check_dict = {}

# Loop through the business units
for bus_unit in campus_bu_list:

    barg_units = ['HX', 'RX', 'TX']

    # start the excel writer outside the loop
    with pd.ExcelWriter(f'{bus_unit}_tcs_step_scale_03202026_v2.xlsx', engine='openpyxl') as writer:

# Loop through the bargaining units
      for barg_unit in barg_units:

        campus_bu = job_details_dict[bus_unit][barg_unit]
        # save the job titles in the bargaining unit in a list from the first layer of the dictionary
        curr_bu = pd.DataFrame(campus_bu).T.index.to_list()

        # Create a list to save all the resulting DataFrames in the current campus' bargaining unit
        campus_bu_list = []

        print(f"currently processing {bus_unit}'s {barg_unit}")


# loop through teh job titles in the current bargaining unit
        for title in curr_bu:
          data = campus_bu[title]
          # Correctly construct header_row as a single-row DataFrame with named columns
          header_row = pd.DataFrame([{
              'Title': data['Title'],
              'Job Code': data['Job Code'][0],
              'Exempt Status': data['Exempt Status'][0],
              'OnCall Eligibility Status': data['OnCall Eligibility Status'][0],
              'Shift Differential Status': data['Shift Differential Status'][0]
          }])


          steps = [int(k) for k in data['Steps']] if data['Half Steps Status'] == 'N' else data['Half Steps']

          # Apparently, some titles have multiple same steps and half steps. Need to manually audit them
          if len(steps) == len(data['Hourly Rates']) == len(data['Monthly Rates']) == len(data['Annual Rates']):
            hourly_rates_row = pd.DataFrame([{'Row_Type': 'Hourly Rates', **{f'Step_{s}': f'${r:,.2f}' for s, r in zip(steps, data['Hourly Rates'])}}])
            monthly_rates_row = pd.DataFrame([{'Row_Type': 'Monthly Rates', **{f'Step_{s}': f'${r:,.2f}' for s, r in zip(steps, data['Monthly Rates'])}}])
            annual_rates_row = pd.DataFrame([{'Row_Type': 'Annual Rates', **{f'Step_{s}': f'${r:,.2f}' for s, r in zip(steps, data['Annual Rates'])}}])
            final_df = pd.concat([header_row,  hourly_rates_row, monthly_rates_row, annual_rates_row], ignore_index=True)

            trailing_empty_row = pd.DataFrame([[None] * len(final_df.columns)] * 3, columns=final_df.columns)
            final_df = pd.concat([final_df, trailing_empty_row], ignore_index=True)
            campus_bu_list.append(final_df)

          else:
            manual_check_dict = {
            print(f"In {bus_unit}, {title} has {len(steps)} steps, {len(data['Hourly Rates'])} HOURLY, \n{len(data['Monthly Rates'])} MONTHLY, and {len(data['Annual Rates'])} ANNUAL")


          for ind in range(len(campus_bu_list)):
            campus_bu_list[ind].to_excel(writer, sheet_name=barg_unit, startrow=ind*7, index=False)





In BKCMP, SYS ADM 3 TX has 19 steps, 38 HOURLY, 
38 MONTHLY, and 38 ANNUAL
In DVCMP, SYS ADM 3 TX has 18 steps, 36 HOURLY, 
36 MONTHLY, and 36 ANNUAL
In DVMED, CLIN LAB SCI PD has 1 steps, 2 HOURLY, 
2 MONTHLY, and 2 ANNUAL
In IRMED, SYS ADM 3 TX has 33 steps, 44 HOURLY, 
44 MONTHLY, and 44 ANNUAL
In LACMP, AUDIOLOGIST EX has 15 steps, 30 HOURLY, 
30 MONTHLY, and 30 ANNUAL
In LACMP, AUDIOLOGIST SR EX has 15 steps, 30 HOURLY, 
30 MONTHLY, and 30 ANNUAL
In LACMP, BEHAVIORAL SVC 1 has 15 steps, 30 HOURLY, 
30 MONTHLY, and 30 ANNUAL
In LACMP, CASE MGR EX has 14 steps, 28 HOURLY, 
28 MONTHLY, and 28 ANNUAL
In LACMP, CHILD DEV ASC SR has 15 steps, 30 HOURLY, 
30 MONTHLY, and 30 ANNUAL
In LACMP, CHILD LIFE SPEC 3 has 15 steps, 30 HOURLY, 
30 MONTHLY, and 30 ANNUAL
In LACMP, CHILD LIFE TEACHER 1 has 15 steps, 30 HOURLY, 
30 MONTHLY, and 30 ANNUAL
In LACMP, CHILD LIFE TEACHER 2 has 15 steps, 30 HOURLY, 
30 MONTHLY, and 30 ANNUAL
In LACMP, CHILD LIFE TEACHER 3 has 15 steps, 30 HOURLY, 
30 MONTHL

In [ ]:
# Now output the UCOP and LBNL with only RX and TX unit
# take out UCOP and LBNL to prevent the code from crashing due to not having "HX" in those business units
remaining_bu_list = ['UCOP1', 'LBNL1']

for bus_unit in remaining_bu_list:

  barg_units = ['RX', 'TX']

  # start the excel writer outside the loop
  with pd.ExcelWriter(f'{bus_unit}_tcs_step_scale_03202026_v2.xlsx', engine='openpyxl') as writer:

    for barg_unit in barg_units:

      campus_bu = job_details_dict[bus_unit][barg_unit]
      # save the job titles in the bargaining unit in a list from the first layer of the dictionary
      curr_bu = pd.DataFrame(campus_bu).T.index.to_list()

      # Create a list to save all the resulting DataFrames in the current campus' bargaining unit
      campus_bu_list = []

    # initiate the excel writer


      for title in curr_bu:
        data = campus_bu[title]
        # Correctly construct header_row as a single-row DataFrame with named columns
        header_row = pd.DataFrame([{
            'Title': data['Title'],
            'Job Code': data['Job Code'][0],
            'Exempt Status': data['Exempt Status'][0],
            'OnCall Eligibility Status': data['OnCall Eligibility Status'][0],
            'Shift Differential Status': data['Shift Differential Status'][0]
        }])


        steps = [int(k) for k in data['Steps']] if data['Half Steps Status'] == 'N' else data['Half Steps']

        # Apparently, some titles have multiple same steps and half steps. Need to manually audit them
        if len(steps) == len(data['Hourly Rates']) == len(data['Monthly Rates']) == len(data['Annual Rates']):
          hourly_rates_row = pd.DataFrame([{'Row_Type': 'Hourly Rates', **{f'Step_{s}': f'${r:,.2f}' for s, r in zip(steps, data['Hourly Rates'])}}])
          monthly_rates_row = pd.DataFrame([{'Row_Type': 'Monthly Rates', **{f'Step_{s}': f'${r:,.2f}' for s, r in zip(steps, data['Monthly Rates'])}}])
          annual_rates_row = pd.DataFrame([{'Row_Type': 'Annual Rates', **{f'Step_{s}': f'${r:,.2f}' for s, r in zip(steps, data['Annual Rates'])}}])
          final_df = pd.concat([header_row,  hourly_rates_row, monthly_rates_row, annual_rates_row], ignore_index=True)

          trailing_empty_row = pd.DataFrame([[None] * len(final_df.columns)] * 3, columns=final_df.columns)
          final_df = pd.concat([final_df, trailing_empty_row], ignore_index=True)
          campus_bu_list.append(final_df)

        else:
          print(f"In {curr_bu}, {title} has {len(steps)} steps, {len(data['Hourly Rates'])} HOURLY, \n{len(data['Monthly Rates'])} MONTHLY, and {len(data['Annual Rates'])} ANNUAL")


        for ind in range(len(campus_bu_list)):
          campus_bu_list[ind].to_excel(writer, sheet_name=barg_unit, startrow=ind*7, index=False)



